# 10. 음성 — 받아쓰기(Whisper) · 읽어 주기(Supertonic)

- **받아쓰기(STT)**: Whisper small (INT8) — `openvino_genai.WhisperPipeline`. 16kHz 모노 소리를 넣으면 글이 나온다.
- **읽어 주기(TTS)**: Supertonic — ONNX 모델 **4개**를 onnxruntime 으로 차례로 돌린다.
  길이 예측 → 글 인코딩 → 잡음 제거 반복(확산) → 보코더(소리 만들기). 코드는 프로그램 폴더의 `speech_routes.py` 의 `_Supertonic`.

## 1. 준비

In [ ]:
import os
import time
from pathlib import Path

import cv2
import numpy as np
import openvino as ov
from PIL import Image
from IPython.display import display

ROOT = Path(os.environ.get("EDGE_ROOT", "..")).resolve()          # 프로그램 폴더 (lab.bat 이 알려 준다)
MODELS = Path(os.environ.get("EDGE_MODELS", ROOT / "models"))      # 모델 폴더
core = ov.Core()
print("모델 폴더:", MODELS)
print("장치:", core.available_devices)

import sys
import sounddevice as sd
from IPython.display import Audio

SR = 16000

## 2. 녹음 — 4초

In [ ]:
SECONDS = 4
print("말하세요 ...")
rec = sd.rec(int(SECONDS * SR), samplerate=SR, channels=1, dtype="float32")
sd.wait()
audio = rec.reshape(-1)
print(f"녹음 {len(audio) / SR:.1f}초 · 가장 큰 소리 {np.abs(audio).max():.2f} (0.05 보다 작으면 마이크를 확인)")
display(Audio(audio, rate=SR))

## 3. 받아쓰기 — Whisper

In [ ]:
import openvino_genai as og

STT_DIR = MODELS / "stt"
DEVICE = "GPU" if any(d.startswith("GPU") for d in core.available_devices) else "CPU"
t0 = time.perf_counter()
whisper = og.WhisperPipeline(str(STT_DIR), DEVICE)
print(f"올림 {time.perf_counter() - t0:.1f}초 ({DEVICE})")

t0 = time.perf_counter()
text = str(whisper.generate(audio, language="<|ko|>", task="transcribe")).strip()
print(f"받아쓰기: {text}   ({(time.perf_counter() - t0) * 1000:.0f} ms)")

## 4. 읽어 주기 — Supertonic (ONNX 4개)

In [ ]:
sys.path.insert(0, str(ROOT))                      # 프로그램 폴더의 speech_routes.py 를 읽는다
import speech_routes

tts = speech_routes._Supertonic(speech_routes.TTS_DIR)
for name, sess in [("길이 예측", tts.dp), ("글 인코딩", tts.enc), ("잡음 제거", tts.est), ("보코더", tts.voc)]:
    print(f"{name:6s} 입력 {[i.name for i in sess.get_inputs()]}")
print("목소리:", speech_routes.VOICES, "| 표본율:", tts.sr)

In [ ]:
t0 = time.perf_counter()
wav = tts.say("안녕하세요. 이 목소리는 이 컴퓨터 안에서 만들었어요.", lang="ko", voice="F1", steps=8, speed=1.05)
dt = time.perf_counter() - t0
print(f"{len(wav) / tts.sr:.1f}초 분량을 {dt:.2f}초에 만듦 (실시간의 {len(wav) / tts.sr / dt:.1f}배)")
display(Audio(wav, rate=tts.sr))

## 5. 이어 붙이기 — 들은 말을 다른 목소리로 되읽기

In [ ]:
if text:
    wav2 = tts.say(text, lang="ko", voice="M1")
    display(Audio(wav2, rate=tts.sr))

## 해 볼 것
1. `steps` 를 2 · 16 으로 바꿔 속도와 음질을 비교해 보자 (잡음 제거 반복 횟수).
2. 목소리 10개(`F1`~`F5`, `M1`~`M5`)를 한 문장으로 모두 들어 보자.
3. 받아쓰기에서 `language="<|en|>"` 로 영어를 해 보자.
4. `sd.play(wav, tts.sr)` 로 노트북 밖(스피커)으로 바로 틀어 보자.